In [ ]:
pip install --upgrade torch ultralytics

In [ ]:
import numpy, cv2, os, pandas,random
import matplotlib.pyplot as plt
from ultralytics import YOLO
from PIL import Image, ImageDraw

In [ ]:
os.makedirs('/kaggle/working/images', exist_ok=True)

yolo = YOLO("/kaggle/input/request-007/segmentation-G1.pt", verbose=False)

In [ ]:
def format_converter(width, height, x1, x2, y1, y2):
    dwidth = (1.0 / width)
    dheight = (1.0 / height)
    x = (((x1 + x2) / 2.0) * dwidth)
    y = (((y1 + y2) / 2.0) * dheight)
    wid = ((x2 - x1) * dwidth)
    hig = ((y2 - y1) * dheight)
    if wid < 0:
        wid = wid * -1
    if hig < 0:
        hig = hig * -1
    return x, y, wid, hig

In [ ]:
bias, l_pre, counter = 4, 5, 100000

lizard_png = ['/kaggle/input/request-007/pngwing.com.png',
              '/kaggle/input/request-007/AdobeStock_1455860250_Preview.png',
              '/kaggle/input/request-007/AdobeStock_535697344_Preview.png']

files = (['/kaggle/input/indoor-object-detection/train/images', 
         '/kaggle/input/indoor-object-detection/test/images',
         '/kaggle/input/indoor-object-detection/valid/images',
         '/kaggle/input/indoor-training-set-its-residestandard/clear',
         '/kaggle/input/indoor-training-set-its-residestandard/hazy',
         '/kaggle/input/indoor-climbing-gym-hold-segmentation/bh',
         '/kaggle/input/indoor-climbing-gym-hold-segmentation/bh-phone',
         '/kaggle/input/indoor-climbing-gym-hold-segmentation/sm'] + 
         [f'/kaggle/input/indoor-images/Images/{i}' for i in os.listdir('/kaggle/input/indoor-images/Images')])

In [ ]:
print(f"Total folders : {len(files)}")
for file in files:
        for i in os.listdir(file):
            data_s, y_coor = Image.open(f'{file}/{i}'), '' 
            data_lab = open(f'train/labels/{counter}.txt', 'w')
            width, height = data_s.size
            print(width, height)
            if width>500 and height>500:
                
                results, true_counter = yolo(data_s, verbose=False), 0
                for result in results:
                    res= result.boxes.xyxy.cpu().detach().numpy()
                    if len(res) > 0:
                        for cod, c, acc in zip(result.masks.xy, result.boxes.cls, result.boxes.conf):
                            data_l = Image.open(random.choice(lizard_png))
                            
                            h_s, w_s = data_s.size
                            p_w = (w_s/100)*l_pre
                            w_l, h_l = data_l.size
                            w_l, h_l = round(p_w), round((p_w/w_l)*h_l)
                            
                            data_l = data_l.resize((w_l, h_l)).rotate(random.randint(0, 360))
                            if acc > 0.5:
                                x_c, y_c, count = 0, 0, 0
                                for x, y in cod:
                                    x_c += x
                                    y_c += y
                                    count += 1
                                    
                                if c == 1:
                                    cod_x, cod_y = round(x_c/count), round(y_c/count)-bias
                                    data_s.paste(data_l, (cod_x, cod_y), data_l)
                                    x, y, wid, hig = format_converter(h_s, w_s, cod_x, (cod_x+w_l), 
                                                                      cod_y, (cod_y+h_l))
                                    if x>1:
                                        continue
                                    elif y>1:
                                        continue
                                    elif wid>1:
                                        continue
                                    elif hig>1:
                                        continue
                                    else:
                                        y_coor += f'0 {x} {y} {wid} {hig}\n'
                                    true_counter += 1

                    if true_counter > 1:
                        data_s.save(f'train/images/d_{counter}.jpg')
                        data_lab.write(y_coor)
                    counter += 1